# Customer Churn Prediction
**Dataset:** IBM Telco Customer Churn (via Kaggle)  
**Author:** Fikri Firstly Arrasyid Hawe  
**Goal:** Build a classification model to predict which telecom customers are likely to churn.

---
### Setup
Run `pip install kagglehub scikit-learn pandas matplotlib seaborn` before starting.  
Make sure you have a Kaggle API key at `~/.kaggle/kaggle.json`.

In [ ]:
# Install dependencies (run once)
# !pip install kagglehub scikit-learn pandas matplotlib seaborn

In [ ]:
import kagglehub
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Download dataset
path = kagglehub.dataset_download('blastchar/telco-customer-churn')
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Exploratory Data Analysis

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#c8a87a', '#1a1a1a'], startangle=90)
axes[0].set_title('Churn Distribution')
sns.countplot(data=df, x='Churn', palette=['#c8a87a', '#1a1a1a'], ax=axes[1])
axes[1].set_title('Churn Count')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly charges vs churn
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette=['#e8d5b7', '#c8a87a'], ax=axes[0])
axes[0].set_title('Monthly Charges by Churn')
sns.boxplot(data=df, x='Churn', y='tenure', palette=['#e8d5b7', '#c8a87a'], ax=axes[1])
axes[1].set_title('Tenure by Churn')
plt.tight_layout()
plt.show()

## 2. Feature Engineering & Preprocessing

In [ ]:
df_model = df.drop(columns=['customerID'])
le = LabelEncoder()
for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = le.fit_transform(df_model[col])

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. Model Training

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
print('=== Logistic Regression ===')
print(classification_report(y_test, lr_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, lr.predict_proba(X_test)[:,1]):.4f}')

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print('=== Random Forest ===')
print(classification_report(y_test, rf_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}')

## 4. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
top15 = importances.tail(15)
top15.plot(kind='barh', color='#c8a87a', figsize=(10, 6))
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 5. Conclusions

- **Random Forest outperforms Logistic Regression** with higher ROC-AUC score
- **Tenure, MonthlyCharges, and TotalCharges** are the strongest predictors of churn
- Customers with **high monthly charges and short tenure** are most at risk
- **Business recommendation:** Target retention campaigns at customers with < 12 months tenure and monthly charges > $65